In [ ]:
import cv2
import numpy as np
import os
import json

# =========================
# 경로 설정
# =========================
input_path = "texture_00.png"
json_path = "./parts_location/parts_bbox.json"

output_dir = "./parts"
os.makedirs(output_dir, exist_ok=True)

# 선택 파츠가 빠진 원본 이미지를 parts 폴더 안에 others.png로 저장
removed_output_path = os.path.join(output_dir, "others.png")

# 수정된 bbox 정보 저장
corrected_json_path = os.path.join(output_dir, "parts_bbox_corrected.json")

# 기존 JSON이 4096 기준이면 그대로 둠
# 만약 네 원본이 4096이면 자동으로 scale = 1이 됨
JSON_CANVAS_WIDTH = 4096
JSON_CANVAS_HEIGHT = 4096

ALPHA_THRESHOLD = 0
MIN_AREA = 20

# =========================
# part 번호 -> 새 이름 매칭
# =========================
rename_map = {
    "part_005": "cloth_neck_left.png",
    "part_006": "cloth_neck_right.png",
    "part_008": "hairband_left.png",
    "part_009": "hairband_right.png",

    "part_010": "arm_left.png",
    "part_011": "arm_right.png",
    "part_012": "cloth_neck_back.png",
    "part_015": "hairband.png",

    "part_016": "body_right.png",
    "part_017": "body_left.png",
    "part_018": "face.png",

    "part_023": "body_1_right.png",
    "part_024": "body_1_left.png",
    "part_030": "neck.png",

    "part_054": "hair_1.png",
    "part_055": "hair_back_2.png",
    "part_056": "hair_2.png",

    "part_057": "hair_shadow_1.png",
    "part_058": "hair_shadow_2.png",
    "part_059": "hair_shadow_3.png",
    "part_060": "hair_shadow_4.png",
    "part_061": "hair_shadow_5.png",
    "part_062": "hair_shadow_6.png",

    "part_063": "hair_3.png",
    "part_064": "hair_4.png",
    "part_065": "hair_shadow_7.png",

    "part_068": "hair_5.png",
    "part_069": "hair_6.png",
    "part_070": "hair_7.png",
    "part_071": "hair_back.png",

    "part_120": "eye_left.png",
    "part_121": "eye_right.png",

    "part_128": "part_128.png",
    "part_129": "part_129.png",
    "part_130": "part_130.png",
    "part_131": "part_131.png",
}

target_ids = set(rename_map.keys())

# =========================
# 함수
# =========================
def bbox_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter
    if union == 0:
        return 0

    return inter / union


def scale_bbox(bbox, sx, sy):
    x1, y1, x2, y2 = bbox
    return [
        int(round(x1 * sx)),
        int(round(y1 * sy)),
        int(round(x2 * sx)),
        int(round(y2 * sy)),
    ]


# =========================
# 원본 이미지 읽기
# =========================
img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)

if img is None:
    raise FileNotFoundError("이미지를 찾을 수 없습니다.")

if len(img.shape) != 3 or img.shape[2] != 4:
    raise ValueError("이 코드는 투명 PNG 즉 RGBA/BGRA 이미지 기준입니다.")

h, w = img.shape[:2]
print("원본 이미지 크기:", w, h)

# =========================
# 기존 JSON 읽기
# =========================
with open(json_path, "r", encoding="utf-8") as f:
    old_parts = json.load(f)

json_max_x = max(part["bbox"][2] for part in old_parts)
json_max_y = max(part["bbox"][3] for part in old_parts)

# JSON bbox가 현재 이미지보다 크면 4096 기준이라고 보고 자동 축소
if json_max_x > w or json_max_y > h:
    sx = w / JSON_CANVAS_WIDTH
    sy = h / JSON_CANVAS_HEIGHT
else:
    sx = 1.0
    sy = 1.0

print("좌표 스케일:", sx, sy)

# =========================
# 원본에서 실제 연결 파츠 다시 찾기
# =========================
alpha = img[:, :, 3]
mask = (alpha > ALPHA_THRESHOLD).astype(np.uint8) * 255

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    mask,
    connectivity=8
)

components = []

for label in range(1, num_labels):
    x, y, bw, bh, area = stats[label]

    if area < MIN_AREA:
        continue

    components.append({
        "label": label,
        "bbox": [int(x), int(y), int(x + bw), int(y + bh)],
        "width": int(bw),
        "height": int(bh),
        "area": int(area),
    })

print("찾은 연결 파츠 수:", len(components))

# =========================
# 결과 이미지 준비
# =========================
# 선택 파츠가 제거된 원본 이미지
remaining_img = img.copy()

corrected_parts = []

# =========================
# 원하는 파츠만 다시 추출
# =========================
for old_part in old_parts:
    part_id = old_part["id"]

    if part_id not in target_ids:
        continue

    new_file_name = rename_map[part_id]

    # 기존 JSON bbox를 현재 이미지 크기에 맞춤
    expected_bbox = scale_bbox(old_part["bbox"], sx, sy)

    # 현재 원본 이미지에서 가장 비슷한 실제 연결 파츠 찾기
    best_component = None
    best_iou = 0

    for comp in components:
        iou = bbox_iou(expected_bbox, comp["bbox"])
        if iou > best_iou:
            best_iou = iou
            best_component = comp

    if best_component is None:
        print("매칭 실패:", part_id)
        continue

    label = best_component["label"]
    x1, y1, x2, y2 = best_component["bbox"]

    # 실제 파츠 영역 crop
    crop = img[y1:y2, x1:x2].copy()

    # 이 component에 해당하는 부분만 남기기
    component_mask = (labels[y1:y2, x1:x2] == label)

    # 원본 alpha 유지하면서 component 밖은 투명 처리
    crop_alpha = crop[:, :, 3].copy()
    crop[:, :, 3] = np.where(component_mask, crop_alpha, 0)

    # 실제 표시되는 픽셀 수
    real_area = int(np.count_nonzero(crop[:, :, 3] > ALPHA_THRESHOLD))

    # 파츠 저장
    save_path = os.path.join(output_dir, new_file_name)
    cv2.imwrite(save_path, crop)

    # 원본에서 해당 파츠 제거
    remain_region = remaining_img[y1:y2, x1:x2]
    remain_region[component_mask] = [0, 0, 0, 0]
    remaining_img[y1:y2, x1:x2] = remain_region

    corrected_info = {
        "id": part_id,
        "old_file": old_part["file"],
        "new_file": new_file_name,
        "bbox": [int(x1), int(y1), int(x2), int(y2)],
        "width": int(x2 - x1),
        "height": int(y2 - y1),
        "area": real_area,
        "bbox_area": int((x2 - x1) * (y2 - y1)),
        "match_iou": round(float(best_iou), 4),
        "old_json_bbox": old_part["bbox"],
        "scaled_expected_bbox": expected_bbox,
    }

    corrected_parts.append(corrected_info)

    print(
        f"{part_id}.png -> {new_file_name} | "
        f"bbox={corrected_info['bbox']} | "
        f"width={corrected_info['width']} | "
        f"height={corrected_info['height']} | "
        f"area={corrected_info['area']} | "
        f"iou={corrected_info['match_iou']}"
    )

# =========================
# 결과 저장
# =========================
with open(corrected_json_path, "w", encoding="utf-8") as f:
    json.dump(corrected_parts, f, ensure_ascii=False, indent=2)

# 선택한 파츠들이 제거된 원본 이미지만 저장
cv2.imwrite(removed_output_path, remaining_img)

print()
print("완료")
print("추출 파츠 저장 폴더:", output_dir)
print("수정된 bbox json:", corrected_json_path)
print("선택 파츠 제거된 원본:", removed_output_path)

원본 이미지 크기: 4096 4096
좌표 스케일: 1.0 1.0
찾은 연결 파츠 수: 177
part_005.png -> cloth_neck_left.png | bbox=[1882, 46, 2009, 225] | width=127 | height=179 | area=7807 | iou=1.0
part_006.png -> cloth_neck_right.png | bbox=[2553, 47, 2661, 218] | width=108 | height=171 | area=7224 | iou=1.0
part_008.png -> hairband_left.png | bbox=[2890, 80, 3369, 315] | width=479 | height=235 | area=62788 | iou=1.0
part_009.png -> hairband_right.png | bbox=[3399, 91, 3888, 313] | width=489 | height=222 | area=62828 | iou=1.0
part_010.png -> arm_left.png | bbox=[1780, 212, 2146, 592] | width=366 | height=380 | area=74273 | iou=1.0
part_011.png -> arm_right.png | bbox=[2491, 220, 2865, 600] | width=374 | height=380 | area=74971 | iou=1.0
part_012.png -> cloth_neck_back.png | bbox=[2211, 271, 2394, 412] | width=183 | height=141 | area=20837 | iou=1.0
part_015.png -> hairband.png | bbox=[3200, 357, 3552, 473] | width=352 | height=116 | area=9960 | iou=0.9915
part_016.png -> body_right.png | bbox=[2337, 415, 2630, 772] 